In [1]:
!pip install -qU langchain langchain-chroma langchain-community langchain-huggingface chromadb pypdf sentence-transformers rapidocr-onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 771.4 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 1.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 51.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 41.2 MB/s et

In [9]:
!pip install pymupdf 

ERROR: Could not find a version that satisfies the requirement langchain_pypdf (from versions: none)
ERROR: No matching distribution found for langchain_pypdf


In [18]:
import fitz  # PyMuPDF
from pathlib import Path
from rapidocr_onnxruntime import RapidOCR
from langchain_core.documents import Document

pdf_folder_path = "/kaggle/input/datasets/jiyajain23/aim-docs" 
chroma_save_path = "/kaggle/working/chroma_db/"
ocr_engine = RapidOCR()

MIN_CHARS_THRESHOLD = 20  # below this, treat the page as scanned/image-only

def load_pdfs_with_ocr_fallback(pdf_folder_path: str) -> list[Document]:
    documents = []
    pdf_paths = sorted(Path(pdf_folder_path).glob("*.pdf"))
    print(f"Found {len(pdf_paths)} PDF files to process.\n")
    for pdf_path in pdf_paths:
        print(f" Processing: {pdf_path.name}...")
        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        for page_num, page in enumerate(doc):
            text = page.get_text().strip()
        
            if len(text) < MIN_CHARS_THRESHOLD:
                # Likely a scanned page — rasterize and OCR it
                print(f"   └─ Page {page_num + 1}/{total_pages}: Running RapidOCR...")
                pix = page.get_pixmap(dpi=150)
                img_bytes = pix.tobytes("png")
                result, _ = ocr_engine(img_bytes)
                text = "\n".join(line[1] for line in result) if result else ""
                source_type = "ocr"
            else:
                source_type = "digital"

            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": str(pdf_path),
                        "page": page_num,
                        "extraction_method": source_type,
                    },
                )
            )
        print(f"   ✅ Completed {pdf_path.name} ({total_pages} pages)\n")
        doc.close()

    return documents

In [19]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


print("Step 1: Loading Documents...")
raw_documents = load_pdfs_with_ocr_fallback(pdf_folder_path)

print(f"-> Loaded {len(raw_documents)} pages from the PDFs.")

print("\nStep 2: Chunking Text...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100, # 100 char overlap ensures context isn't lost between chunks
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(raw_documents)
print(f"-> Split documents into {len(chunks)} searchable chunks.")

print("\nStep 3: Initializing Embedding Model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("\nStep 4: Creating Vector Database (This may take a few minutes)...")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=chroma_save_path
)

print(f"\n Phase 1 Complete! Vector database saved to {chroma_save_path}")
print("You can now download this folder from the Kaggle Output section later.")

Step 1: Loading Documents...
Found 7 PDF files to process.

 Processing: ATMC 02 of 2025 SAR Training for Non-RCC ATS Personnel.pdf...
   ✅ Completed ATMC 02 of 2025 SAR Training for Non-RCC ATS Personnel.pdf (4 pages)

 Processing: ATMC 02 of 2026.pdf...
   ✅ Completed ATMC 02 of 2026.pdf (8 pages)

 Processing: ATMC 02_2024.pdf...
   └─ Page 1/4: Running RapidOCR...
   └─ Page 2/4: Running RapidOCR...
   └─ Page 3/4: Running RapidOCR...
   └─ Page 4/4: Running RapidOCR...
   ✅ Completed ATMC 02_2024.pdf (4 pages)

 Processing: ATMC 05 of 2025.pdf...
   ✅ Completed ATMC 05 of 2025.pdf (7 pages)

 Processing: ATMC 06 of 2025-Data Collection for Airspace safety Monitoring.pdf...
   └─ Page 1/25: Running RapidOCR...
   └─ Page 2/25: Running RapidOCR...
   └─ Page 3/25: Running RapidOCR...
   └─ Page 4/25: Running RapidOCR...
   └─ Page 5/25: Running RapidOCR...
   └─ Page 6/25: Running RapidOCR...
   └─ Page 7/25: Running RapidOCR...
   └─ Page 8/25: Running RapidOCR...
   └─ Page 9/25: 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Step 4: Creating Vector Database (This may take a few minutes)...

 Phase 1 Complete! Vector database saved to /kaggle/working/chroma_db/
You can now download this folder from the Kaggle Output section later.


In [20]:
!pip install -qU langchain-groq langchain-classic rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.1 MB/s eta 0:00:00a 0:00:01


In [21]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from kaggle_secrets import UserSecretsClient

# BM25 needs the raw chunk list, not the vector store
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 6  # widen the net since reranking will narrow it back down

vector_retriever = vector_db.as_retriever(search_kwargs={"k": 6})

# Leaning toward vector search (0.75) since it's more semantically reliable;
# BM25 (0.25) still contributes for exact-term/acronym queries
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.25, 0.75]
)

groq_api_key = UserSecretsClient().get_secret("groq-api-key")
llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0)

In [22]:
from sentence_transformers import CrossEncoder
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List
from langchain_core.documents import Document

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

class RerankRetriever(BaseRetriever):
    base_retriever: BaseRetriever
    top_n: int = 3

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        docs = self.base_retriever.invoke(query)
        if not docs:
            return []
        pairs = [(query, d.page_content) for d in docs]
        scores = reranker.predict(pairs)
        reranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, score in reranked[: self.top_n]]

reranked_hybrid_retriever = RerankRetriever(base_retriever=hybrid_retriever, top_n=3)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [35]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.messages import HumanMessage, AIMessage

# 1. Prompt to reformulate follow-up questions using chat history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and the latest user question which might reference "
               "context in the chat history, reformulate it into a standalone question. "
               "Do NOT answer the question, just reformulate it if needed, otherwise return it as is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

history_aware_retriever = create_history_aware_retriever(llm, reranked_hybrid_retriever, contextualize_prompt)

# 2. QA prompt that incorporates context and history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based only on the following context. "
               "If the answer isn't in the context, say you don't know.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

conv_combine_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, conv_combine_chain)

# 3. Memory tracking helper
chat_history = []

def ask(question, show_sources=True):
    result = conversational_rag_chain.invoke({"input": question, "chat_history": chat_history})
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result["answer"]))
    if show_sources:
        print("Sources:", [(d.metadata.get("source","?").split("/")[-1], d.metadata.get("page","?")) for d in result["context"]])
    return result["answer"]

# 4. Test follow-up queries!
print(ask("Eligibility criteria for ATCO"))

Sources: [('ATMC 05 of 2025.pdf', 1), ('ATMC 05 of 2025.pdf', 0), ('ATMC 05 of 2025.pdf', 2)]
According to the given context, the eligibility criteria for ATCOs for Surveillance Control Course (SCC) are as follows:

1. The ATCO shall hold a valid Class III medical assessment. However, when the Medical Assessment from DGCA is awaited, the ATCO shall be in possession of a medical certificate (CA 35) issued by the Designated Medical Examiner (DME).

2. The officer shall have a minimum of one year of experience at the station of posting after acquiring the relevant rating(s), as per Para 4.2.1(a) or (b), whichever is applicable.

3. In the case of ATCOs who were previously posted at a surveillance station and were eligible for nomination as per criteria mentioned in Para 4.1 and 4.2 of this ATMC at that station, they shall regain eligibility once they have revalidated or acquired the relevant rating(s).


In [38]:
import pandas as pd

# Fill in with real questions + known-correct facts from your manual
eval_set = [
    {"question": "What is an Abnormal Runway Contact (ARC)?",
     "expected_keywords": ["runway", "contact"]},
    {"question": "What does ATS stand for?",
     "expected_keywords": ["air traffic service"]},
    {"question":"What data does ATS units need to collect",
     "expected_keywords":["traffic sample data","large height deviation"]},
    {"question":"Eligibility criteria for ATCO",
     "expected_keywords":["Class III medical assessment","one year of experience"]},
    {"question":"The SAR presentation in the Refresher Training for ATCOs should cover what topics?",
     "expected_keywords":["sar system components","sar cooperation"]},
    {"question":"ATS units",
     "expected_keywords":["flight information service","air traffic advisory service"]},
]

def evaluate(chain, eval_set):
    results = []
    for item in eval_set:
        # ⭐️ FIX: Added "chat_history": [] to satisfy the history-aware retriever
        result = chain.invoke({
            "input": item["question"], 
            "chat_history": []
        })
        
        answer = result["answer"]
        answer_lower = answer.lower()
        
        # Check for keyword hits
        hits = [kw for kw in item["expected_keywords"] if kw.lower() in answer_lower]
        
        results.append({
            "question": item["question"],
            "answer": answer,
            "expected_keywords": item["expected_keywords"],
            "keywords_found": hits,
            "score": len(hits) / len(item["expected_keywords"])
        })
        
    return pd.DataFrame(results)

eval_df = evaluate(conversational_rag_chain , eval_set)

print(f"Average score: {eval_df['score'].mean():.2%}\n")
display(eval_df[["question", "score", "keywords_found"]]) # using display() looks much cleaner in Kaggle notebooks!

Average score: 100.00%



,question,score,keywords_found
0,What is an Abnormal Runway Contact (ARC)?,1.0,"[runway, contact]"
1,What does ATS stand for?,1.0,[air traffic service]
2,What data does ATS units need to collect,1.0,"[traffic sample data, large height deviation]"
3,Eligibility criteria for ATCO,1.0,"[Class III medical assessment, one year of exp..."
4,The SAR presentation in the Refresher Training...,1.0,"[sar system components, sar cooperation]"
5,ATS units,1.0,"[flight information service, air traffic advis..."


In [26]:
!zip -r chroma_db.zip /kaggle/working/chroma_db

  adding: kaggle/working/chroma_db/ (stored 0%)
  adding: kaggle/working/chroma_db/chroma.sqlite3 (deflated 57%)
  adding: kaggle/working/chroma_db/01b5ba13-93d8-4790-9847-d7f0efed6d83/ (stored 0%)
  adding: kaggle/working/chroma_db/01b5ba13-93d8-4790-9847-d7f0efed6d83/length.bin (deflated 98%)
  adding: kaggle/working/chroma_db/01b5ba13-93d8-4790-9847-d7f0efed6d83/data_level0.bin (deflated 12%)
  adding: kaggle/working/chroma_db/01b5ba13-93d8-4790-9847-d7f0efed6d83/header.bin (deflated 57%)
  adding: kaggle/working/chroma_db/01b5ba13-93d8-4790-9847-d7f0efed6d83/link_lists.bin (deflated 80%)
  adding: kaggle/working/chroma_db/01b5ba13-93d8-4790-9847-d7f0efed6d83/index_metadata.pickle (deflated 46%)
